In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [5]:
df = pd.read_csv("../dataset/raw/UNSW_NB15_training-set.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (82332, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [6]:
selected_features = [
    "dur",
    "proto",
    "state",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sttl",
    "dttl"
]

df = df[selected_features + ["attack_cat"]]

print("Selected Columns:")
print(df.columns.tolist())

df.head()

Selected Columns:
['dur', 'proto', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'attack_cat']


,dur,proto,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,attack_cat
0,0.000011,udp,INT,2,0,496,0,90909.0902,254,0,Normal
1,0.000008,udp,INT,2,0,1762,0,125000.0003,254,0,Normal
2,0.000005,udp,INT,2,0,1068,0,200000.0051,254,0,Normal
3,0.000006,udp,INT,2,0,900,0,166666.6608,254,0,Normal
4,0.000010,udp,INT,2,0,2126,0,100000.0025,254,0,Normal


In [7]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
dur           0
proto         0
state         0
spkts         0
dpkts         0
sbytes        0
dbytes        0
rate          0
sttl          0
dttl          0
attack_cat    0
dtype: int64


In [8]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = [
    "proto",
    "state"
]

encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])
    encoders[column] = encoder

print(df.head())

        dur  proto  state  spkts  dpkts  sbytes  dbytes         rate  sttl  \
0  0.000011    117      4      2      0     496       0   90909.0902   254   
1  0.000008    117      4      2      0    1762       0  125000.0003   254   
2  0.000005    117      4      2      0    1068       0  200000.0051   254   
3  0.000006    117      4      2      0     900       0  166666.6608   254   
4  0.000010    117      4      2      0    2126       0  100000.0025   254   

   dttl attack_cat  
0     0     Normal  
1     0     Normal  
2     0     Normal  
3     0     Normal  
4     0     Normal  


In [9]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(encoders["proto"], "../models/proto_encoder.pkl")
joblib.dump(encoders["state"], "../models/state_encoder.pkl")

print("✓ Encoders saved successfully!")

✓ Encoders saved successfully!


In [7]:
df["label"] = df["attack_cat"].apply(
    lambda x: 0 if x == "Normal" else 1
)

df.drop("attack_cat", axis=1, inplace=True)

print(df.head())

        dur  proto  state  spkts  dpkts  sbytes  dbytes         rate  sttl  \
0  0.000011    117      4      2      0     496       0   90909.0902   254   
1  0.000008    117      4      2      0    1762       0  125000.0003   254   
2  0.000005    117      4      2      0    1068       0  200000.0051   254   
3  0.000006    117      4      2      0     900       0  166666.6608   254   
4  0.000010    117      4      2      0    2126       0  100000.0025   254   

   dttl  label  
0     0      0  
1     0      0  
2     0      0  
3     0      0  
4     0      0  


In [8]:
X = df.drop("label", axis=1)
y = df["label"]

print("Feature Shape:", X.shape)
print("Label Shape:", y.shape)

Feature Shape: (82332, 10)
Label Shape: (82332,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Testing Set :", X_test.shape)

Training Set: (65865, 10)
Testing Set : (16467, 10)


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling Complete")

Scaling Complete


In [11]:
import numpy as np

np.save("../dataset/processed/X_train.npy", X_train)
np.save("../dataset/processed/X_test.npy", X_test)

np.save("../dataset/processed/y_train.npy", y_train)
np.save("../dataset/processed/y_test.npy", y_test)

print("Training and Testing data saved.")

Training and Testing data saved.


In [12]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("Scaler saved successfully.")

Scaler saved successfully.


In [13]:
feature_df = pd.DataFrame(
    {"Feature": selected_features}
)

feature_df.to_csv(
    "../dataset/processed/selected_features.csv",
    index=False
)

print("Selected features saved.")

Selected features saved.


In [14]:
joblib.dump(
    encoders,
    "../models/encoders.pkl"
)

print("Encoders saved.")

Encoders saved.


In [15]:
print("=" * 50)
print("PREPROCESSING COMPLETED")
print("=" * 50)

print("Training Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)

print("\nSelected Features:")
for feature in selected_features:
    print("-", feature)

print("\nFiles Saved:")
print("✔ X_train.npy")
print("✔ X_test.npy")
print("✔ y_train.npy")
print("✔ y_test.npy")
print("✔ scaler.pkl")
print("✔ encoders.pkl")
print("✔ selected_features.csv")

PREPROCESSING COMPLETED
Training Samples : (65865, 10)
Testing Samples  : (16467, 10)

Selected Features:
- dur
- proto
- state
- spkts
- dpkts
- sbytes
- dbytes
- rate
- sttl
- dttl

Files Saved:
✔ X_train.npy
✔ X_test.npy
✔ y_train.npy
✔ y_test.npy
✔ scaler.pkl
✔ encoders.pkl
✔ selected_features.csv


In [16]:
print(df.columns.tolist())

['dur', 'proto', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'label']
